Challenge: List IP addresses + sum of requests

In [ ]:
from pyspark.sql import SparkSession, functions as func, Row
from pyspark.sql.functions import to_timestamp
import re



spark = SparkSession.builder.appName('LogChallenge1').getOrCreate()

logs = spark.read.text('access_log.txt', header=False, inferSchema=True)

# print('Here is our inferred scheme')

# logs.printSchema()

# Returning a Row object. And then manually cast the fields to the correct type
# A better way to do this is to infer a schema, supposedly 

log_pattern = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "([^"]*)" "([^"]*)"'

def parse_log_line(line):
    match = re.match(log_pattern, line)
    fields = match.groups()
    return Row(
        ip=str(fields[0]),
        client_identity=str(fields[1]),
        user_id=str(fields[2]),
        timestamp=str(fields[3]),
        method=str(fields[4]),
        path=str(fields[5]),
        protocol=str(fields[6]),
        status=int(fields[7]),
        bytes_sent=int(fields[8]) if fields[8] != '-' else 0,
        referrer=str(fields[9]),
        user_agent=str(fields[10])
    )


# The below is taken from a different file to use as a guide vvv
rdd = spark.sparkContext.textFile('./access_log.csv')
row_rdd = rdd.map(parse_log_line)
schemaLogs = spark.createDataFrame(logs).cache() # improves performance if need to do multiple things with this object
# The cache() makes it so that the program doesn't need to rebuild the DataFrame 
schemaLogs.createOrReplaceTempView('logs')

schemaLogs = schemaLogs.withColumn("timestamp", to_timestamp("timestamp", "dd/MMM/yyyy:HH:mm:ss Z"))

schemaLogs.printSchema()
schemaLogs.select("ip").show()

spark.stop()

# adults = spark.sql("SELECT * FROM people WHERE age >= 18") # Makes use of view made from schemaPeople

# for p in adults.collect():
#     print(p)

# schemaPeople.groupBy('age').count().orderBy('age').show() # MAkes use of schemaPeople again


# r'\W+' is a regex that looks for any non-word character (anything that is not a letter, digit, or underscore) and splits the line into words based on that
# So each line is split into words, and then the explode() function creates a new row for each word
# words = inputDF.select(func.explode(func.split(inputDF.value, r'\W+')).alias('word'))
# wordsWithoutEmptyStrs = words.filter(words.word != '')


# lowercaseWords = wordsWithoutEmptyStrs.select(func.lower(wordsWithoutEmptyStrs.word).alias('word'))
# wordCountsSorted = lowercaseWords.groupBy('word').count().sort('count', ascending=False)

# wordCountsSorted.show() #

# spark.stop()
